In [ ]:
# 保存图片
import os
import matplotlib.pyplot as plt
import scanpy as sc
def save_fig(plt, out_dir = os.getcwd(), file_name=None, fig_size={'w':5, 'h':5}):
    plt.tight_layout()
    try:
        plt.gcf().set_size_inches(fig_size['w'], fig_size['h'])
    except:
        pass
    plt.savefig(os.path.join(out_dir, f'{file_name}.png'), bbox_inches = 'tight', pad_inches = 0.1)
    plt.savefig(os.path.join(out_dir, f'{file_name}.pdf'), bbox_inches = 'tight', pad_inches = 0.1)
    plt.clf()

# 加载亚群细分的函数
import sys
sys.path.append('/lvdata/wzb/pipline/Fun_py')
from subcluster import recluster
from find_marker import find_marker_gene
from draw_group import draw_group
sys.path.append('/lvdata/wzb/pipline/function/GSVA')
import GSVA
# os.chdir('/lvdata/wzb/scRNA/FW2023-656/2024.01.02/')
import pandas as pd
import gseapy as gp

In [ ]:
# 先读取h5ad文件按照比较的组别拍讯并进行gsea分析
adata = sc.read_h5ad('/lvdata/wzb/scRNA/FW2024-225_03/2024.12.03/Result.h5ad')
adata.obs.group = pd.Categorical(adata.obs['group'], categories=["iMCD", "NC"], ordered=True)
indices = adata.obs.sort_values(['cell_type_new', 'group']).index
adata = adata[indices,:]
bdata = adata[adata.obs.cell_type == "CCL Monocytes"].copy()
bdata = bdata.raw.to_adata()
res = gp.gsea(data=bdata.to_df().T, # row -> genes, column-> samples
        gene_sets="/lvdata/wzb/scRNA/FW2024-225_03/2024.12.03/gmt/h.all.v2024.1.Hs.symbols.gmt",
        cls=bdata.obs.group,
        permutation_num=200,
        permutation_type='phenotype',
        outdir="/lvdata/wzb/scRNA/FW2024-225_03/2024.12.03/HALLMARK/CCL Monocytes",
        method='s2n', # signal_to_noise
        min_size = 1,
        max_size=100000,
        threads= 32)

In [ ]:
# 读取结果中的结果文件用于后续作图
df = pd.read_csv('/lvdata/wzb/scRNA/FW2024-225_03/2024.12.03/HALLMARK/CCL Monocytes/gseapy.phenotype.gsea.report.csv')

In [ ]:
# 计算每个基因集里面的颜色，并提供不同p值颜色对应列表
df['counts'] = [len(x) for x in df['Lead_genes'].str.split(';')]
def re_color(e):
    if e <= 0.01:
        return '**'
    if (e > 0.01) & (e <= 0.05):
        return '*'
    if e > 0.05:
        return 'ns'
df['color'] = [re_color(x) for x in df['FDR q-val']]
custom_palette = {
    '**': 'red',  # 红色
    '*': 'green',  # 绿色
    'ns': 'blue'   # 蓝色
}

In [ ]:

# 绘制散点图
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import linregress
sns.scatterplot(
        data=df,
        x='FDR q-val',
        y='NES',
        size='counts',
        hue='color',
        sizes=(10, 150), # 调整尺寸范围
        palette=custom_palette
    )
# 反转横轴
plt.gca().invert_xaxis()
# 设置X轴间隔
def frange(start, stop, step):
    while start <= stop:
        yield start
        start += step
plt.xticks(ticks=[round(x, 2) for x in list(frange(0, 1.1, 0.1))])
# 添加红色虚线参考线
plt.axvline(x=0.25, color='red', linestyle='--', label='Reference Line')

# 设置 y 轴范围
plt.ylim(0, 3)
save_fig(plt,out_dir='/lvdata/wzb/scRNA/FW2024-225_03/2024.12.03/GSEA_scatterplot',file_name='HALLMARK_CCLMonocytes_iMCDvsNC',fig_size={'w':5,'h':7})